In [1]:
!pip install scanpy anndata torch torchvision torchaudio
!pip install pot mygene gseapy networkx scikit-learn

  Using cached torchvision-0.27.0-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached torchaudio-2.11.0-cp312-cp312-win_amd64.whl.metadata (6.9 kB)
  Using cached torch-2.12.0-cp312-cp312-win_amd64.whl.metadata (31 kB)
Using cached torchvision-0.27.0-cp312-cp312-win_amd64.whl (4.0 MB)
Using cached torch-2.12.0-cp312-cp312-win_amd64.whl (123.0 MB)
Using cached torchaudio-2.11.0-cp312-cp312-win_amd64.whl (328 kB)

   ---------------------------------------- 0/3 [torchaudio]
   ---------------------------------------- 0/3 [torchaudio]
   ---------------------------------------- 0/3 [torchaudio]
   ---------------------------------------- 0/3 [torchaudio]
   ---------------------------------------- 0/3 [torchaudio]
   ---------------------------------------- 0/3 [torchaudio]
   ---------------------------------------- 0/3 [torchaudio]
   ---------------------------------------- 0/3 [torchaudio]
   ---------------------------------------- 0/3 [torchaudio]
   -------------------------

In [8]:
import scanpy as sc
import pandas as pd
import numpy as np

# Example: load preprocessed Adamson h5ad
# Replace path with your dataset
adata = sc.read_h5ad("adamson_biolord.h5ad")

print(adata)
print("Cells:", adata.n_obs)
print("Genes:", adata.n_vars)

AnnData object with n_obs × n_vars = 65899 × 5060
    obs: 'condition', 'cell_type', 'dose_val', 'control', 'condition_name', 'split1', 'subgroup1', 'split2', 'subgroup2', 'split3', 'subgroup3', 'split4', 'subgroup4', 'split5', 'subgroup5', 'perturbation'
    var: 'gene_name'
    uns: 'non_dropout_gene_idx', 'non_zeros_gene_idx', 'pert2neighbor', 'rank_genes_groups_cov_all', 'top_non_dropout_de_20', 'top_non_zero_de_20'
    obsm: 'perturbation_neighbors', 'perturbation_neighbors_pca'
Cells: 65899
Genes: 5060


In [9]:
# Example: load preprocessed Adamson h5ad
# Replace path with your dataset
adata = sc.read_h5ad("adamson_biolord.h5ad")

print(adata)
print("Cells:", adata.n_obs)
print("Genes:", adata.n_vars)

# Perturbation labels
print(adata.obs['condition'].value_counts().head())


AnnData object with n_obs × n_vars = 65899 × 5060
    obs: 'condition', 'cell_type', 'dose_val', 'control', 'condition_name', 'split1', 'subgroup1', 'split2', 'subgroup2', 'split3', 'subgroup3', 'split4', 'subgroup4', 'split5', 'subgroup5', 'perturbation'
    var: 'gene_name'
    uns: 'non_dropout_gene_idx', 'non_zeros_gene_idx', 'pert2neighbor', 'rank_genes_groups_cov_all', 'top_non_dropout_de_20', 'top_non_zero_de_20'
    obsm: 'perturbation_neighbors', 'perturbation_neighbors_pca'
Cells: 65899
Genes: 5060
condition
ctrl            24263
IER3IP1+ctrl     1267
YIPF5+ctrl       1182
SCYL1+ctrl       1024
HSPA5+ctrl       1002
Name: count, dtype: int64


In [4]:
 #PHASE 2 — PREPROCESSING
# =========================================================
# QC
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

# Normalize total counts
sc.pp.normalize_total(adata, target_sum=1e4)

# Log transform
sc.pp.log1p(adata)

# Highly Variable Genes
sc.pp.highly_variable_genes(adata, n_top_genes=5000)
adata = adata[:, adata.var.highly_variable]

print("After HVG:", adata.shape)

After HVG: (65895, 5000)


In [5]:
# PHASE 3 — SPLIT CONTROL / PERTURBED
# =========================================================
control = adata[adata.obs['condition'] == 'ctrl']
perturbed = adata[adata.obs['condition'] != 'ctrl']

print("Control:", control.shape)
print("Perturbed:", perturbed.shape)


Control: (24259, 5000)
Perturbed: (41636, 5000)


In [6]:
#BUILD GENE REPRESENTATIONS
# =========================================================
genes = list(adata.var_names)

# ---- (A) Placeholder LLM embeddings (replace with GenePT) ----
llm_dim = 3072
llm_embeddings = {g: np.random.randn(llm_dim) for g in genes}

# ---- (B) GO similarity ----
go_dim = len(genes)
go_embeddings = {g: np.random.rand(go_dim) for g in genes}

# ---- (C) STRING similarity ----
string_embeddings = {g: np.random.rand(go_dim) for g in genes}

In [7]:
# PHASE 5 — COMBINE MULTI-SOURCE EMBEDDINGS
# =========================================================
def get_gene_embedding(gene):
    llm = llm_embeddings[gene]
    go = go_embeddings[gene]
    st = string_embeddings[gene]
    return np.concatenate([llm, go, st])

gene_embeddings = {g: get_gene_embedding(g) for g in genes}

embed_dim = len(next(iter(gene_embeddings.values())))
print("Combined embedding dim:", embed_dim)


Combined embedding dim: 13072


In [8]:
print("Unique condition labels:")
print(adata.obs['condition'].unique()[:50])

Unique condition labels:
['CREB1+ctrl', 'ctrl', 'ZNF326+ctrl', 'BHLHE40+ctrl', 'DDIT3+ctrl', ..., 'TARS+ctrl', 'ASCC3+ctrl', 'EIF2B2+ctrl', 'SARS+ctrl', 'ARHGAP22+ctrl']
Length: 50
Categories (82, object): ['AARS+ctrl', 'ARHGAP22+ctrl', 'ASCC3+ctrl', 'ATP5B+ctrl', ..., 'XRN1+ctrl', 'YIPF5+ctrl', 'ZNF326+ctrl', 'ctrl']


In [9]:
# =========================================================
# ROBUST CONTROL DETECTION FOR ADAMSON
# =========================================================

conditions = adata.obs['condition'].astype(str)

# Common control keywords
control_keywords = ['ctrl', 'control', 'nt', 'non-targeting']

# Mark control rows
control_mask = conditions.str.lower().apply(
    lambda x: any(k in x for k in control_keywords)
)

# Split
control = adata[control_mask].copy()
perturbed = adata[~control_mask].copy()

print("Control cells:", control.shape)
print("Perturbed cells:", perturbed.shape)

# Optional: inspect examples
print("Control labels sample:")
print(control.obs['condition'].value_counts().head(10))

Control cells: (65895, 5000)
Perturbed cells: (0, 5000)
Control labels sample:
condition
ctrl            24259
IER3IP1+ctrl     1267
YIPF5+ctrl       1182
SCYL1+ctrl       1024
HSPA5+ctrl       1002
SEC61A1+ctrl      962
IARS2+ctrl        923
GBF1+ctrl         786
HSPA9+ctrl        718
SAMM50+ctrl       672
Name: count, dtype: int64


In [10]:
# =========================================================
# STRICT CONTROL / PERTURBED SPLIT
# =========================================================

conditions = adata.obs['condition'].astype(str).str.lower()

# Pure controls only
pure_control_labels = ['ctrl', 'ctrl+ctrl']

control_mask = conditions.isin(pure_control_labels)

control = adata[control_mask].copy()
perturbed = adata[~control_mask].copy()

print("Control cells:", control.shape)
print("Perturbed cells:", perturbed.shape)

print("\nControl label counts:")
print(control.obs['condition'].value_counts())

print("\nPerturbed sample labels:")
print(perturbed.obs['condition'].value_counts().head(10))

Control cells: (24259, 5000)
Perturbed cells: (41636, 5000)

Control label counts:
condition
ctrl    24259
Name: count, dtype: int64

Perturbed sample labels:
condition
IER3IP1+ctrl    1267
YIPF5+ctrl      1182
SCYL1+ctrl      1024
HSPA5+ctrl      1002
SEC61A1+ctrl     962
IARS2+ctrl       923
GBF1+ctrl        786
HSPA9+ctrl       718
SAMM50+ctrl      672
TARS+ctrl        671
Name: count, dtype: int64


In [11]:
# =========================================================
# PHASE 6 — EXTRACT PERTURBED GENE + BUILD TRAINING MATRICES
# =========================================================

import numpy as np
import pandas as pd

# ---------- Parse Adamson condition ----------
def extract_gene(condition):
    """
    Examples:
    IER3IP1+ctrl -> IER3IP1
    ctrl+IER3IP1 -> IER3IP1
    ctrl -> ctrl
    ctrl+ctrl -> ctrl
    """
    parts = str(condition).split("+")
    
    # remove ctrl tokens
    genes = [p for p in parts if p.lower() != "ctrl"]
    
    if len(genes) == 0:
        return "ctrl"
    
    # Adamson single-gene perturbation
    return genes[0]

# Parse all perturbed labels
pert_conditions = perturbed.obs["condition"].astype(str).values
pert_genes = [extract_gene(c) for c in pert_conditions]

print("Sample parsed genes:", pert_genes[:20])

# =========================================================
# KEEP ONLY GENES AVAILABLE IN EMBEDDINGS
# =========================================================
valid_idx = [i for i, g in enumerate(pert_genes) if g in gene_embeddings]

print("Valid perturbation cells:", len(valid_idx), "/", len(pert_genes))

# Filter perturbation matrix
X_pert = perturbed.X
if hasattr(X_pert, "toarray"):
    X_pert = X_pert.toarray()

X_pert = X_pert[valid_idx]

# Valid genes only
pert_genes_valid = [pert_genes[i] for i in valid_idx]

# =========================================================
# MATCH CONTROL CELLS
# =========================================================
# We usually sample equal number of controls
X_control = control.X
if hasattr(X_control, "toarray"):
    X_control = X_control.toarray()

# Random sample controls equal to perturbed count
np.random.seed(42)

control_idx = np.random.choice(
    X_control.shape[0],
    size=len(valid_idx),
    replace=len(valid_idx) > X_control.shape[0]
)

X_control = X_control[control_idx]

# =========================================================
# BUILD PERTURBATION EMBEDDING MATRIX
# =========================================================
P = np.array([gene_embeddings[g] for g in pert_genes_valid])

# =========================================================
# FINAL CHECK
# =========================================================
print("\nFINAL DATA SHAPES")
print("X_control:", X_control.shape)
print("X_pert:", X_pert.shape)
print("P:", P.shape)

# Optional sanity checks
print("\nUnique perturbation genes:", len(set(pert_genes_valid)))
print("Top perturbation genes:")
print(pd.Series(pert_genes_valid).value_counts().head(10))

Sample parsed genes: ['CREB1', 'ZNF326', 'CREB1', 'CREB1', 'BHLHE40', 'BHLHE40', 'CREB1', 'DDIT3', 'ZNF326', 'ZNF326', 'ZNF326', 'ZNF326', 'ZNF326', 'CREB1', 'ZNF326', 'CREB1', 'CREB1', 'CREB1', 'BHLHE40', 'ZNF326']
Valid perturbation cells: 0 / 41636

FINAL DATA SHAPES
X_control: (0, 5000)
X_pert: (0, 5000)
P: (0,)

Unique perturbation genes: 0
Top perturbation genes:
Series([], Name: count, dtype: int64)


In [12]:
# =========================================================
# ROOT CAUSE:
# gene_embeddings keys do NOT match Adamson symbols
# Usually because:
# 1. Lowercase vs uppercase mismatch
# 2. Ensembl IDs vs gene symbols
# 3. Missing perturbed genes after HVG filtering
# =========================================================

# FIRST — inspect both sides
print("Sample perturbation genes:")
print(pert_genes[:20])

print("\nSample gene_embedding keys:")
print(list(gene_embeddings.keys())[:20])

print("\nSample adata.var_names:")
print(list(adata.var_names[:20]))

Sample perturbation genes:
['CREB1', 'ZNF326', 'CREB1', 'CREB1', 'BHLHE40', 'BHLHE40', 'CREB1', 'DDIT3', 'ZNF326', 'ZNF326', 'ZNF326', 'ZNF326', 'ZNF326', 'CREB1', 'ZNF326', 'CREB1', 'CREB1', 'CREB1', 'BHLHE40', 'ZNF326']

Sample gene_embedding keys:
['ENSG00000228463', 'ENSG00000272438', 'ENSG00000230699', 'ENSG00000223764', 'ENSG00000187634', 'ENSG00000188290', 'ENSG00000187608', 'ENSG00000162571', 'ENSG00000186891', 'ENSG00000186827', 'ENSG00000221978', 'ENSG00000272455', 'ENSG00000215915', 'ENSG00000272106', 'ENSG00000197530', 'ENSG00000189409', 'ENSG00000169885', 'ENSG00000271806', 'ENSG00000182873', 'ENSG00000269554']

Sample adata.var_names:
['ENSG00000228463', 'ENSG00000272438', 'ENSG00000230699', 'ENSG00000223764', 'ENSG00000187634', 'ENSG00000188290', 'ENSG00000187608', 'ENSG00000162571', 'ENSG00000186891', 'ENSG00000186827', 'ENSG00000221978', 'ENSG00000272455', 'ENSG00000215915', 'ENSG00000272106', 'ENSG00000197530', 'ENSG00000189409', 'ENSG00000169885', 'ENSG00000271806', 

In [13]:
# =========================================================
# FIX: Adamson perturbation labels = GENE SYMBOLS
# adata.var_names / embeddings = ENSEMBL IDs
# We must map ENSEMBL -> SYMBOL
# =========================================================

# Check available gene metadata columns
print("adata.var columns:")
print(adata.var.columns.tolist())

adata.var columns:
['gene_name', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm']


In [14]:
# =========================================================
# PERFECT — use adata.var['gene_name']
# Build SYMBOL -> ENSEMBL mapping
# =========================================================

# Example:
# ENSG00000123456 -> CREB1
# We need:
# CREB1 -> ENSG00000123456

# Build mapping
symbol_to_ensembl = {}

for ensembl_id, gene_symbol in zip(adata.var_names, adata.var["gene_name"]):
    if pd.notna(gene_symbol):
        symbol_to_ensembl[str(gene_symbol).upper()] = str(ensembl_id)

print("Total mapped genes:", len(symbol_to_ensembl))

# Sample checks
for g in ["CREB1", "ZNF326", "BHLHE40", "DDIT3"]:
    print(g, "->", symbol_to_ensembl.get(g, "NOT FOUND"))

# =========================================================
# CONVERT PERTURBATION SYMBOLS -> ENSEMBL
# =========================================================
pert_genes_ensembl = []

for g in pert_genes:
    ensembl = symbol_to_ensembl.get(str(g).upper(), None)
    pert_genes_ensembl.append(ensembl)

# Valid indices
valid_idx = [i for i, g in enumerate(pert_genes_ensembl) if g is not None]

print("\nValid mapped perturbations:", len(valid_idx), "/", len(pert_genes))

# =========================================================
# FILTER MATRICES
# =========================================================
X_pert = perturbed.X
if hasattr(X_pert, "toarray"):
    X_pert = X_pert.toarray()

X_pert = X_pert[valid_idx]

# Matched control cells
X_control = control.X
if hasattr(X_control, "toarray"):
    X_control = X_control.toarray()

np.random.seed(42)

control_idx = np.random.choice(
    X_control.shape[0],
    size=len(valid_idx),
    replace=len(valid_idx) > X_control.shape[0]
)

X_control = X_control[control_idx]

# =========================================================
# BUILD EMBEDDING MATRIX USING ENSEMBL IDS
# =========================================================
valid_pert_genes = [pert_genes_ensembl[i] for i in valid_idx]

P = np.array([gene_embeddings[g] for g in valid_pert_genes])

# =========================================================
# FINAL CHECK
# =========================================================
print("\nFINAL SHAPES")
print("X_control:", X_control.shape)
print("X_pert:", X_pert.shape)
print("P:", P.shape)

print("\nUnique valid perturbation genes:", len(set(valid_pert_genes)))

# Top genes
mapped_symbols = [pert_genes[i] for i in valid_idx]
print("\nTop mapped perturbations:")
print(pd.Series(mapped_symbols).value_counts().head(10))

Total mapped genes: 5000
CREB1 -> ENSG00000118260
ZNF326 -> ENSG00000162664
BHLHE40 -> ENSG00000134107
DDIT3 -> ENSG00000175197

Valid mapped perturbations: 36767 / 41636

FINAL SHAPES
X_control: (36767, 5000)
X_pert: (36767, 5000)
P: (36767, 13072)

Unique valid perturbation genes: 73

Top mapped perturbations:
IER3IP1    1267
YIPF5      1182
SCYL1      1024
HSPA5      1002
SEC61A1     962
GBF1        786
HSPA9       718
XRN1        621
SRPRB       596
UFM1        591
Name: count, dtype: int64


In [5]:
# =========================================================
# FULL REBUILD AFTER KERNEL RESET
# This reconstructs EVERYTHING from adata
# =========================================================

import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

# =========================================================
# STEP 1 — STRICT CONTROL / PERTURBED SPLIT
# =========================================================
adata = sc.read_h5ad('adamson_biolord.h5ad')

conditions = adata.obs["condition"].astype(str).str.lower()

pure_control_labels = ["ctrl", "ctrl+ctrl"]

control_mask = conditions.isin(pure_control_labels)

control = adata[control_mask].copy()
perturbed = adata[~control_mask].copy()

print("Control:", control.shape)
print("Perturbed:", perturbed.shape)

# =========================================================
# STEP 2 — SYMBOL -> ENSEMBL MAP
# =========================================================
symbol_to_ensembl = {}

for ensembl_id, gene_symbol in zip(adata.var_names, adata.var["gene_name"]):
    if pd.notna(gene_symbol):
        symbol_to_ensembl[str(gene_symbol).upper()] = str(ensembl_id)

print("Mapped genes:", len(symbol_to_ensembl))

# =========================================================
# STEP 3 — PARSE ADAMSON CONDITIONS
# =========================================================
def extract_gene(condition):
    parts = str(condition).split("+")
    genes = [p for p in parts if p.lower() != "ctrl"]

    if len(genes) == 0:
        return "ctrl"

    return genes[0]

pert_conditions = perturbed.obs["condition"].astype(str).values
pert_genes = [extract_gene(c) for c in pert_conditions]

# =========================================================
# STEP 4 — SYMBOL -> ENSEMBL
# =========================================================
pert_genes_ensembl = [
    symbol_to_ensembl.get(str(g).upper(), None)
    for g in pert_genes
]

valid_idx = np.array([
    i for i, g in enumerate(pert_genes_ensembl)
    if g is not None
])

print("Valid perturbation cells:", len(valid_idx))

valid_pert_genes = [pert_genes_ensembl[i] for i in valid_idx]

# =========================================================
# STEP 5 — MATRICES
# =========================================================
X_pert = perturbed.X
if hasattr(X_pert, "toarray"):
    X_pert = X_pert.toarray()

X_pert = X_pert[valid_idx].astype(np.float32)

X_control_full = control.X
if hasattr(X_control_full, "toarray"):
    X_control_full = X_control_full.toarray()

X_control_full = X_control_full.astype(np.float32)

# Match control count
np.random.seed(42)

control_idx = np.random.choice(
    X_control_full.shape[0],
    size=len(valid_idx),
    replace=len(valid_idx) > X_control_full.shape[0]
)

X_control = X_control_full[control_idx]

print("X_control:", X_control.shape)
print("X_pert:", X_pert.shape)

# =========================================================
# STEP 6 — REBUILD GENE EMBEDDINGS IF LOST
# SIMPLE VERSION: use gene expression identity vectors
# (Replace later with STRING + GO + LLM)
# =========================================================
if "gene_embeddings" not in globals():
    print("\nRebuilding fallback embeddings...")
    
    genes = list(adata.var_names)

    gene_embeddings = {}

    for i, g in enumerate(genes):
        vec = np.zeros(len(genes), dtype=np.float32)
        vec[i] = 1.0
        gene_embeddings[g] = vec

# =========================================================
# STEP 7 — BUILD P
# =========================================================
P = np.array(
    [gene_embeddings[g] for g in valid_pert_genes],
    dtype=np.float32
)

print("Original P:", P.shape)

# =========================================================
# STEP 8 — PCA REDUCTION
# =========================================================
pca_dim = 256

pca = PCA(n_components=pca_dim, random_state=42)

P_reduced = pca.fit_transform(P).astype(np.float32)

print("Reduced P:", P_reduced.shape)
print("Explained variance:", round(
    pca.explained_variance_ratio_.sum(), 4
))

# =========================================================
# STEP 9 — GENE SPLIT
# =========================================================
unique_genes = np.array(sorted(list(set(valid_pert_genes))))

train_genes, temp_genes = train_test_split(
    unique_genes,
    test_size=0.30,
    random_state=42
)

val_genes, test_genes = train_test_split(
    temp_genes,
    test_size=0.50,
    random_state=42
)

gene_array = np.array(valid_pert_genes)

train_idx = np.where(np.isin(gene_array, train_genes))[0]
val_idx   = np.where(np.isin(gene_array, val_genes))[0]
test_idx  = np.where(np.isin(gene_array, test_genes))[0]

print("\nCells:")
print("Train:", len(train_idx))
print("Val:", len(val_idx))
print("Test:", len(test_idx))

# =========================================================
# STEP 10 — DATASET
# =========================================================
class AdamsonDataset(Dataset):
    def __init__(self, Xc, Xp, P, indices):
        self.Xc = Xc
        self.Xp = Xp
        self.P = P
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]

        return (
            torch.tensor(self.Xc[i], dtype=torch.float32),
            torch.tensor(self.Xp[i], dtype=torch.float32),
            torch.tensor(self.P[i], dtype=torch.float32)
        )

# =========================================================
# STEP 11 — DATALOADERS
# =========================================================
batch_size = 64

train_loader = DataLoader(
    AdamsonDataset(X_control, X_pert, P_reduced, train_idx),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    AdamsonDataset(X_control, X_pert, P_reduced, val_idx),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    AdamsonDataset(X_control, X_pert, P_reduced, test_idx),
    batch_size=batch_size,
    shuffle=False
)

print("\nSUCCESS")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Control: (24263, 5060)
Perturbed: (41636, 5060)
Mapped genes: 5060
Valid perturbation cells: 41636
X_control: (41636, 5060)
X_pert: (41636, 5060)

Rebuilding fallback embeddings...
Original P: (41636, 5060)
Reduced P: (41636, 256)
Explained variance: 1.0

Cells:
Train: 28982
Val: 5418
Test: 7236

SUCCESS
Train batches: 453
Val batches: 85
Test batches: 114


In [14]:
# USE YOUR EXISTING VARIABLES:
# control = control AnnData
# perturbed = perturbed AnnData

# =========================
# STEP 1 — CHECK PERTURBED METADATA
# =========================
print("=" * 60)
print("PERTURBED OBS COLUMNS")
print("=" * 60)

print(perturbed.obs.columns.tolist())

# Preview metadata
print("\nSample rows:")
print(perturbed.obs.head())

# =========================
# STEP 2 — AUTO-DETECT LIKELY PERTURBATION COLUMN
# =========================
possible_cols = [
    col for col in perturbed.obs.columns
    if any(keyword in col.lower() for keyword in
           ["gene", "target", "condition", "perturb", "guide"])
]

print("\nPossible perturbation columns:")
print(possible_cols)

# =========================
# STEP 3 — SET COLUMN
# Replace after checking output
# =========================
perturbation_col = possible_cols[0]   # adjust if needed

# Extract perturbation labels
perturbation_genes = perturbed.obs[perturbation_col].astype(str).values

print("\nTotal perturbed cells:", len(perturbation_genes))
print("Unique perturbations:", len(set(perturbation_genes)))
print("Sample perturbations:", pd.Series(perturbation_genes).unique()[:20])

# =========================
# STEP 4 — CHOOSE UNSEEN GENE
# =========================
# Pick from printed sample
unseen_gene = pd.Series(perturbation_genes).unique()[0]

print("\nUsing unseen gene:", unseen_gene)

# =========================
# STEP 5 — HOLDOUT SPLIT
# =========================
train_idx = perturbation_genes != unseen_gene
test_idx  = perturbation_genes == unseen_gene

print("\nUNSEEN GENE SPLIT")
print("=" * 40)
print("Train Cells:", train_idx.sum())
print("Test Cells :", test_idx.sum())

# =========================
# STEP 6 — APPLY TO MATRICES
# Assumes:
# X_control
# X_pert
# already exist
# =========================
Xc_train = X_control[train_idx]
Xp_train = X_pert[train_idx]

Xc_test = X_control[test_idx]
Xp_test = X_pert[test_idx]

print("\nShapes:")
print("Xc_train:", Xc_train.shape)
print("Xp_train:", Xp_train.shape)
print("Xc_test :", Xc_test.shape)
print("Xp_test :", Xp_test.shape)

PERTURBED OBS COLUMNS
['condition', 'cell_type', 'dose_val', 'control', 'condition_name', 'split1', 'subgroup1', 'split2', 'subgroup2', 'split3', 'subgroup3', 'split4', 'subgroup4', 'split5', 'subgroup5', 'perturbation']

Sample rows:
                     condition cell_type dose_val  control  \
cell_barcode                                                 
AAACATACACCGAT-1    CREB1+ctrl   K562(?)      1+1        0   
AAACCGTGCAGCTA-1   ZNF326+ctrl   K562(?)      1+1        0   
AAACCGTGGAACTC-1    CREB1+ctrl   K562(?)      1+1        0   
AAACGCACGAACCT-1    CREB1+ctrl   K562(?)      1+1        0   
AAACGCTGAATCGC-1  BHLHE40+ctrl   K562(?)      1+1        0   

                            condition_name split1      subgroup1 split2  \
cell_barcode                                                              
AAACATACACCGAT-1    K562(?)_CREB1+ctrl_1+1    ood  unseen_single  train   
AAACCGTGCAGCTA-1   K562(?)_ZNF326+ctrl_1+1  train      Train/Val  train   
AAACCGTGGAACTC-1    K562(?)_CR

In [15]:
# PERFECT — NOW USE ACTUAL GENE NAME, NOT FULL "GENE+ctrl"
# Extract clean perturbation gene from condition column
# Example:
# CREB1+ctrl → CREB1

# =========================
# STEP 1 — CLEAN GENE NAME
# =========================
perturbation_genes = (
    perturbed.obs["condition"]
    .astype(str)
    .str.split("+")
    .str[0]
    .values
)

print("=" * 60)
print("CLEANED PERTURBATION GENES")
print("=" * 60)

unique_genes = pd.Series(perturbation_genes).unique()

print("Total perturbed cells:", len(perturbation_genes))
print("Unique genes:", len(unique_genes))
print("Sample genes:", unique_genes[:20])

# =========================
# STEP 2 — CHOOSE SPECIFIC UNSEEN GENE
# =========================
# Replace with any valid gene from sample list
unseen_gene = "CREB1"

if unseen_gene not in set(perturbation_genes):
    print(f"{unseen_gene} not found.")
else:
    # =========================
    # STEP 3 — HOLDOUT SPLIT
    # =========================
    train_idx = perturbation_genes != unseen_gene
    test_idx  = perturbation_genes == unseen_gene

    print("\nUNSEEN GENE HOLDOUT")
    print("=" * 40)
    print("Unseen Gene :", unseen_gene)
    print("Train Cells :", train_idx.sum())
    print("Test Cells  :", test_idx.sum())

    # =========================
    # STEP 4 — SPLIT MATRICES
    # =========================
    Xc_train = X_control[train_idx]
    Xp_train = X_pert[train_idx]

    Xc_test = X_control[test_idx]
    Xp_test = X_pert[test_idx]

    print("\nShapes:")
    print("Xc_train:", Xc_train.shape)
    print("Xp_train:", Xp_train.shape)
    print("Xc_test :", Xc_test.shape)
    print("Xp_test :", Xp_test.shape)

    # =========================
    # STEP 5 — OPTIONAL VALIDATION SPLIT
    # =========================
    from sklearn.model_selection import train_test_split

    train_idx2, val_idx2 = train_test_split(
        np.where(train_idx)[0],
        test_size=0.1,
        random_state=42
    )

    Xc_val = X_control[val_idx2]
    Xp_val = X_pert[val_idx2]

    print("\nValidation:")
    print("Val Cells:", len(val_idx2))
    print("Xc_val:", Xc_val.shape)
    print("Xp_val:", Xp_val.shape)

CLEANED PERTURBATION GENES
Total perturbed cells: 41636
Unique genes: 81
Sample genes: ['CREB1' 'ZNF326' 'BHLHE40' 'DDIT3' 'OST4' 'SEC61A1' 'EIF2B4' 'IER3IP1'
 'YIPF5' 'HARS' 'DAD1' 'DARS' 'IDH3A' 'IARS2' 'SOCS1' 'UFL1' 'DDRGK1'
 'QARS' 'STT3A' 'SAMM50']

UNSEEN GENE HOLDOUT
Unseen Gene : CREB1
Train Cells : 41164
Test Cells  : 472

Shapes:
Xc_train: (41164, 5060)
Xp_train: (41164, 5060)
Xc_test : (472, 5060)
Xp_test : (472, 5060)

Validation:
Val Cells: 4117
Xc_val: (4117, 5060)
Xp_val: (4117, 5060)


In [16]:
# =========================================================
# BUILD GENE EMBEDDINGS WITHOUT PCA
# Pure biological effect embedding:
# Each gene = mean perturbation signature (full 5060D)
#
# gene_to_vec[gene] -> full 5060-dimensional effect vector
# Zero-shot safe
# No dimensionality reduction
# =========================================================

import numpy as np
import pandas as pd

# =========================
# STEP 1 — CLEAN GENE LABELS
# =========================
perturbation_genes = (
    perturbed.obs["condition"]
    .astype(str)
    .str.split("+")
    .str[0]
    .values
)

unique_genes = np.unique(perturbation_genes)

print("=" * 60)
print("BUILDING FULL GENE EFFECT EMBEDDINGS (NO PCA)")
print("=" * 60)

print("Unique genes:", len(unique_genes))

# =========================
# STEP 2 — BUILD GENE EFFECT VECTOR
# =========================
gene_to_vec = {}
valid_genes = []

for gene in unique_genes:
    idx = perturbation_genes == gene

    # Skip tiny groups
    if idx.sum() < 5:
        continue

    # Full perturbation effect
    # Shape: (5060,)
    mean_effect = (X_pert[idx] - X_control[idx]).mean(axis=0)

    gene_to_vec[gene] = mean_effect.astype(np.float32)
    valid_genes.append(gene)

print("Valid genes:", len(valid_genes))

# =========================
# STEP 3 — INSPECT
# =========================
sample_gene = valid_genes[0]

print("\nSample Gene:", sample_gene)
print("Embedding shape:", gene_to_vec[sample_gene].shape)

# =========================
# STEP 4 — REMOVE UNSEEN GENE FROM TRAINING DICTIONARY
# CRITICAL FOR TRUE ZERO-SHOT
# =========================
if unseen_gene in gene_to_vec:
    del gene_to_vec[unseen_gene]

print("\nUnseen gene removed from training dictionary:", unseen_gene not in gene_to_vec)

# =========================
# STEP 5 — BUILD CELL-WISE GENE MATRICES
# =========================
def build_gene_matrix(gene_labels, gene_dict, dim=5060):
    embeddings = []
    missing = 0

    for g in gene_labels:
        if g in gene_dict:
            embeddings.append(gene_dict[g])
        else:
            # unseen gene = zero vector
            embeddings.append(np.zeros(dim, dtype=np.float32))
            missing += 1

    return np.vstack(embeddings), missing

# Labels
train_genes = perturbation_genes[train_idx]
val_genes   = perturbation_genes[val_idx2]
test_genes  = perturbation_genes[test_idx]

# Build matrices
G_train, miss_train = build_gene_matrix(train_genes, gene_to_vec)
G_val,   miss_val   = build_gene_matrix(val_genes, gene_to_vec)
G_test,  miss_test  = build_gene_matrix(test_genes, gene_to_vec)

# =========================
# STEP 6 — RESULTS
# =========================
print("\nGENE EMBEDDING MATRICES")
print("=" * 50)

print("G_train:", G_train.shape, "| Missing:", miss_train)
print("G_val  :", G_val.shape,   "| Missing:", miss_val)
print("G_test :", G_test.shape,  "| Missing:", miss_test)

# =========================
# EXPECTATION:
# G_test missing should equal all unseen cells
# because unseen gene embedding is intentionally absent
# =========================

print("\nExpected unseen test cells:", test_idx.sum())
print("Actual zero-vector test embeddings:", miss_test)

# =========================
# FINAL:
# Xc_train = control state
# G_train  = gene embedding
# Xp_train = target perturbed state
#
# Model Input:
# [Xc_train, G_train]
# =========================

BUILDING FULL GENE EFFECT EMBEDDINGS (NO PCA)
Unique genes: 81
Valid genes: 81

Sample Gene: AARS
Embedding shape: (5060,)

Unseen gene removed from training dictionary: True

GENE EMBEDDING MATRICES
G_train: (41164, 5060) | Missing: 0
G_val  : (4117, 5060) | Missing: 0
G_test : (472, 5060) | Missing: 472

Expected unseen test cells: 472
Actual zero-vector test embeddings: 472


In [17]:
# =========================================================
# GENEFORMER-LITE ZERO-SHOT MODEL
# Control + Gene Embedding -> Predicted Perturbed State
#
# INPUT:
# X_control (5060)
# G_gene    (5060)
#
# OUTPUT:
# X_pert    (5060)
#
# LOSS:
# MSE + Pearson
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# DATASET
# =========================
class GeneDataset(Dataset):
    def __init__(self, Xc, G, Y):
        self.Xc = torch.tensor(Xc, dtype=torch.float32)
        self.G  = torch.tensor(G, dtype=torch.float32)
        self.Y  = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.Xc)

    def __getitem__(self, idx):
        return self.Xc[idx], self.G[idx], self.Y[idx]

# =========================
# DATALOADERS
# =========================
batch_size = 64

train_loader = DataLoader(
    GeneDataset(Xc_train, G_train, Xp_train),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    GeneDataset(Xc_val, G_val, Xp_val),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    GeneDataset(Xc_test, G_test, Xp_test),
    batch_size=batch_size,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

# =========================
# MODEL
# =========================
class GeneFormerLite(nn.Module):
    def __init__(self, input_dim=10120, output_dim=5060):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 4096),
            nn.LayerNorm(4096),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(4096, 2048),
            nn.LayerNorm(2048),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(2048, 1024),
            nn.LayerNorm(1024),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(1024, output_dim)
        )

    def forward(self, xc, g):
        x = torch.cat([xc, g], dim=1)
        return self.net(x)

model = GeneFormerLite().to(device)

# =========================
# LOSS
# =========================
def pearson_corr(pred, target):
    pred_centered = pred - pred.mean(dim=1, keepdim=True)
    target_centered = target - target.mean(dim=1, keepdim=True)

    numerator = (pred_centered * target_centered).sum(dim=1)
    denominator = torch.sqrt(
        (pred_centered ** 2).sum(dim=1) *
        (target_centered ** 2).sum(dim=1) + 1e-8
    )

    return numerator / denominator

def combined_loss(pred, target):
    mse = F.mse_loss(pred, target)
    corr = pearson_corr(pred, target).mean()

    return mse + (1 - corr)

# =========================
# OPTIMIZER
# =========================
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

# =========================
# TRAIN LOOP
# =========================
epochs = 30
best_val = float("inf")

for epoch in range(epochs):

    # -----------------
    # TRAIN
    # -----------------
    model.train()
    train_loss = 0

    for xc, g, y in train_loader:
        xc, g, y = xc.to(device), g.to(device), y.to(device)

        optimizer.zero_grad()

        pred = model(xc, g)

        loss = combined_loss(pred, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # -----------------
    # VALIDATION
    # -----------------
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for xc, g, y in val_loader:
            xc, g, y = xc.to(device), g.to(device), y.to(device)

            pred = model(xc, g)

            loss = combined_loss(pred, y)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    # -----------------
    # SAVE BEST
    # -----------------
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "best_geneformer_lite.pt")

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

print("\nBEST VAL LOSS:", best_val)
print("Saved: best_geneformer_lite.pt")

Device: cpu
Train batches: 644
Val batches: 65
Test batches: 8
Epoch 01/30 | Train Loss: 0.3129 | Val Loss: 0.2496
Epoch 02/30 | Train Loss: 0.2559 | Val Loss: 0.2491
Epoch 03/30 | Train Loss: 0.2524 | Val Loss: 0.2479
Epoch 04/30 | Train Loss: 0.2505 | Val Loss: 0.2474
Epoch 05/30 | Train Loss: 0.2495 | Val Loss: 0.2468
Epoch 06/30 | Train Loss: 0.2487 | Val Loss: 0.2464
Epoch 07/30 | Train Loss: 0.2479 | Val Loss: 0.2459
Epoch 08/30 | Train Loss: 0.2473 | Val Loss: 0.2451
Epoch 09/30 | Train Loss: 0.2467 | Val Loss: 0.2449
Epoch 10/30 | Train Loss: 0.2461 | Val Loss: 0.2439
Epoch 11/30 | Train Loss: 0.2455 | Val Loss: 0.2436
Epoch 12/30 | Train Loss: 0.2448 | Val Loss: 0.2430
Epoch 13/30 | Train Loss: 0.2441 | Val Loss: 0.2426
Epoch 14/30 | Train Loss: 0.2433 | Val Loss: 0.2417
Epoch 15/30 | Train Loss: 0.2426 | Val Loss: 0.2400
Epoch 16/30 | Train Loss: 0.2416 | Val Loss: 0.2401
Epoch 17/30 | Train Loss: 0.2405 | Val Loss: 0.2386
Epoch 18/30 | Train Loss: 0.2396 | Val Loss: 0.2386
E

In [18]:
# =========================================================
# ZERO-SHOT EVALUATION ON UNSEEN GENE (CREB1)
# Evaluates:
# 1. MSE
# 2. Pearson
# 3. R²
# 4. Per-gene correlation
# =========================================================

import torch
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr

# =========================
# LOAD BEST MODEL
# =========================
model = GeneFormerLite().to(device)
model.load_state_dict(torch.load("best_geneformer_lite.pt", map_location=device))
model.eval()

print("Loaded best_geneformer_lite.pt")

# =========================
# PREDICT TEST SET
# =========================
all_preds = []
all_true = []

with torch.no_grad():
    for xc, g, y in test_loader:
        xc, g = xc.to(device), g.to(device)

        pred = model(xc, g)

        all_preds.append(pred.cpu().numpy())
        all_true.append(y.numpy())

# Combine
Y_pred = np.vstack(all_preds)
Y_true = np.vstack(all_true)

print("Prediction shape:", Y_pred.shape)
print("Ground truth shape:", Y_true.shape)

# =========================
# GLOBAL METRICS
# =========================
mse = mean_squared_error(Y_true.flatten(), Y_pred.flatten())
r2 = r2_score(Y_true.flatten(), Y_pred.flatten())

# Cell-wise Pearson
cellwise_corrs = []
for i in range(Y_true.shape[0]):
    try:
        corr, _ = pearsonr(Y_true[i], Y_pred[i])
        if not np.isnan(corr):
            cellwise_corrs.append(corr)
    except:
        pass

mean_cellwise_pearson = np.mean(cellwise_corrs)

# Gene-wise Pearson
gene_corrs = []
for g in range(Y_true.shape[1]):
    try:
        corr, _ = pearsonr(Y_true[:, g], Y_pred[:, g])
        if not np.isnan(corr):
            gene_corrs.append(corr)
    except:
        pass

mean_genewise_pearson = np.mean(gene_corrs)

# =========================
# RESULTS
# =========================
print("\n" + "=" * 60)
print("ZERO-SHOT UNSEEN GENE EVALUATION")
print("=" * 60)

print("Unseen Gene:", unseen_gene)
print("Test Cells :", Y_true.shape[0])

print("\nGLOBAL:")
print("MSE                  :", round(mse, 6))
print("R²                   :", round(r2, 6))

print("\nPEARSON:")
print("Mean Cell-wise       :", round(mean_cellwise_pearson, 6))
print("Mean Gene-wise       :", round(mean_genewise_pearson, 6))

# =========================
# TOP / WORST GENES
# =========================
gene_corrs = np.array(gene_corrs)

top_idx = np.argsort(gene_corrs)[-20:][::-1]
worst_idx = np.argsort(gene_corrs)[:20]

print("\nTOP 20 BEST-PREDICTED GENES:")
for idx in top_idx:
    print(f"GeneIndex {idx}: {gene_corrs[idx]:.4f}")

print("\nTOP 20 WORST-PREDICTED GENES:")
for idx in worst_idx:
    print(f"GeneIndex {idx}: {gene_corrs[idx]:.4f}")

# =========================
# SAVE RESULTS
# =========================
results = {
    "unseen_gene": unseen_gene,
    "mse": mse,
    "r2": r2,
    "mean_cellwise_pearson": mean_cellwise_pearson,
    "mean_genewise_pearson": mean_genewise_pearson
}

import pandas as pd
pd.DataFrame([results]).to_csv(
    f"zero_shot_{unseen_gene}_results.csv",
    index=False
)

print(f"\nSaved: zero_shot_{unseen_gene}_results.csv")

Loaded best_geneformer_lite.pt
Prediction shape: (472, 5060)
Ground truth shape: (472, 5060)


C:\Users\aman\AppData\Local\Temp\ipykernel_3636\3056913333.py:68: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = pearsonr(Y_true[:, g], Y_pred[:, g])



ZERO-SHOT UNSEEN GENE EVALUATION
Unseen Gene: CREB1
Test Cells : 472

GLOBAL:
MSE                  : 0.092426
R²                   : 0.738403

PEARSON:
Mean Cell-wise       : 0.86064
Mean Gene-wise       : 0.000929

TOP 20 BEST-PREDICTED GENES:
GeneIndex 460: 0.4331
GeneIndex 1783: 0.1851
GeneIndex 2163: 0.1591
GeneIndex 2154: 0.1586
GeneIndex 3021: 0.1558
GeneIndex 3160: 0.1550
GeneIndex 3382: 0.1536
GeneIndex 1706: 0.1487
GeneIndex 2834: 0.1477
GeneIndex 1257: 0.1455
GeneIndex 746: 0.1437
GeneIndex 2517: 0.1423
GeneIndex 1520: 0.1347
GeneIndex 1221: 0.1340
GeneIndex 809: 0.1338
GeneIndex 1027: 0.1332
GeneIndex 2400: 0.1330
GeneIndex 1957: 0.1325
GeneIndex 2396: 0.1324
GeneIndex 2835: 0.1315

TOP 20 WORST-PREDICTED GENES:
GeneIndex 1743: -0.1569
GeneIndex 1057: -0.1544
GeneIndex 2697: -0.1517
GeneIndex 305: -0.1471
GeneIndex 252: -0.1448
GeneIndex 650: -0.1385
GeneIndex 752: -0.1347
GeneIndex 3357: -0.1314
GeneIndex 2150: -0.1279
GeneIndex 2676: -0.1279
GeneIndex 3311: -0.1241
GeneIn

In [19]:
# =========================================================
# BUILD BIOLOGICAL EMBEDDING FOR UNSEEN GENE (CREB1)
# Uses:
# 1. GO terms
# 2. STRING interactions
# 3. NCBI / GeneCards prior biology
#
# PRACTICAL VERSION:
# If direct APIs unavailable, approximate CREB1 embedding
# from biologically similar genes in Adamson dataset
#
# IDEA:
# CREB1 embedding = weighted average of nearest known genes
# =========================================================

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# =========================
# STEP 1 — EXISTING GENE EFFECT MATRIX
# gene_to_vec from seen genes only
# =========================
seen_genes = list(gene_to_vec.keys())

print("=" * 60)
print("BIOLOGICAL PROXY EMBEDDING FOR CREB1")
print("=" * 60)

print("Seen genes available:", len(seen_genes))

# =========================
# STEP 2 — MANUAL BIOLOGICAL PRIOR
# Add genes biologically related to CREB1
#
# Examples:
# CREB1 related:
# ATF4, DDIT3, BHLHE40, SOCS1, EIF2B4
# (stress response / transcriptional regulation)
# =========================
candidate_genes = [
    "DDIT3",
    "BHLHE40",
    "SOCS1",
    "EIF2B4",
    "ZNF326"
]

# Keep only valid genes
proxy_genes = [g for g in candidate_genes if g in gene_to_vec]

print("Proxy genes used:", proxy_genes)

# =========================
# STEP 3 — BUILD PROXY EMBEDDING
# Weighted mean (equal weights baseline)
# =========================
if len(proxy_genes) == 0:
    raise ValueError("No proxy genes found.")

proxy_vectors = np.vstack([gene_to_vec[g] for g in proxy_genes])

CREB1_embedding = proxy_vectors.mean(axis=0).astype(np.float32)

print("CREB1 proxy embedding shape:", CREB1_embedding.shape)

# =========================
# STEP 4 — REPLACE ZERO-VECTOR TEST EMBEDDING
# =========================
G_test_bio = np.vstack([
    CREB1_embedding for _ in range(Xc_test.shape[0])
])

print("Updated G_test_bio:", G_test_bio.shape)

# =========================
# STEP 5 — OPTIONAL:
# Compare similarity of CREB1 proxy to known genes
# =========================
all_seen_matrix = np.vstack([gene_to_vec[g] for g in seen_genes])

sims = cosine_similarity(
    CREB1_embedding.reshape(1, -1),
    all_seen_matrix
)[0]

top_idx = np.argsort(sims)[-10:][::-1]

print("\nTop biologically similar seen genes to CREB1 proxy:")
for idx in top_idx:
    print(f"{seen_genes[idx]}: {sims[idx]:.4f}")

# =========================
# STEP 6 — SAVE
# =========================
np.save("CREB1_biological_embedding.npy", CREB1_embedding)

print("\nSaved: CREB1_biological_embedding.npy")

# =========================
# NEXT:
# Replace:
# G_test
# with:
# G_test_bio
#
# Then rerun zero-shot evaluation
# =========================

BIOLOGICAL PROXY EMBEDDING FOR CREB1
Seen genes available: 80
Proxy genes used: ['DDIT3', 'BHLHE40', 'SOCS1', 'EIF2B4', 'ZNF326']
CREB1 proxy embedding shape: (5060,)
Updated G_test_bio: (472, 5060)

Top biologically similar seen genes to CREB1 proxy:
BHLHE40: 0.8637
DDIT3: 0.8171
ZNF326: 0.7848
SARS: 0.1241
HARS: 0.1210
COPZ1: 0.0924
EIF2B4: 0.0836
CARS: 0.0664
AARS: 0.0637
MTHFD1: 0.0459

Saved: CREB1_biological_embedding.npy


In [20]:
# =========================================================
# RE-EVALUATE ZERO-SHOT WITH BIOLOGICAL CREB1 EMBEDDING
# Compare:
# ZERO VECTOR vs BIOLOGICAL PROXY
# =========================================================

import torch
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr
from torch.utils.data import DataLoader, Dataset

# =========================
# DATASET FOR BIO TEST
# =========================
class GeneDataset(Dataset):
    def __init__(self, Xc, G, Y):
        self.Xc = torch.tensor(Xc, dtype=torch.float32)
        self.G  = torch.tensor(G, dtype=torch.float32)
        self.Y  = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.Xc)

    def __getitem__(self, idx):
        return self.Xc[idx], self.G[idx], self.Y[idx]

bio_test_loader = DataLoader(
    GeneDataset(Xc_test, G_test_bio, Xp_test),
    batch_size=64,
    shuffle=False
)

# =========================
# LOAD MODEL
# =========================
model = GeneFormerLite().to(device)
model.load_state_dict(torch.load("best_geneformer_lite.pt", map_location=device))
model.eval()

print("Loaded best_geneformer_lite.pt")

# =========================
# PREDICT
# =========================
all_preds = []
all_true = []

with torch.no_grad():
    for xc, g, y in bio_test_loader:
        xc, g = xc.to(device), g.to(device)

        pred = model(xc, g)

        all_preds.append(pred.cpu().numpy())
        all_true.append(y.numpy())

Y_pred_bio = np.vstack(all_preds)
Y_true = np.vstack(all_true)

# =========================
# METRICS
# =========================
mse_bio = mean_squared_error(Y_true.flatten(), Y_pred_bio.flatten())
r2_bio = r2_score(Y_true.flatten(), Y_pred_bio.flatten())

# Cell-wise Pearson
cell_corrs_bio = []
for i in range(Y_true.shape[0]):
    try:
        corr, _ = pearsonr(Y_true[i], Y_pred_bio[i])
        if not np.isnan(corr):
            cell_corrs_bio.append(corr)
    except:
        pass

mean_cell_bio = np.mean(cell_corrs_bio)

# Gene-wise Pearson
gene_corrs_bio = []
for g in range(Y_true.shape[1]):
    try:
        if np.std(Y_true[:, g]) == 0 or np.std(Y_pred_bio[:, g]) == 0:
            continue
        corr, _ = pearsonr(Y_true[:, g], Y_pred_bio[:, g])
        if not np.isnan(corr):
            gene_corrs_bio.append(corr)
    except:
        pass

mean_gene_bio = np.mean(gene_corrs_bio)

# =========================
# ORIGINAL BASELINE
# Fill these from prior run
# =========================
baseline_mse = 0.092426
baseline_r2 = 0.738403
baseline_cell = 0.86064
baseline_gene = 0.000929

# =========================
# RESULTS
# =========================
print("\n" + "=" * 70)
print("CREB1 ZERO-SHOT COMPARISON")
print("=" * 70)

print(f"{'Metric':<20}{'Zero Vector':<15}{'Bio Proxy':<15}{'Delta':<15}")
print("-" * 70)

print(f"{'MSE':<20}{baseline_mse:<15.6f}{mse_bio:<15.6f}{(baseline_mse - mse_bio):<15.6f}")
print(f"{'R²':<20}{baseline_r2:<15.6f}{r2_bio:<15.6f}{(r2_bio - baseline_r2):<15.6f}")
print(f"{'Cell Pearson':<20}{baseline_cell:<15.6f}{mean_cell_bio:<15.6f}{(mean_cell_bio - baseline_cell):<15.6f}")
print(f"{'Gene Pearson':<20}{baseline_gene:<15.6f}{mean_gene_bio:<15.6f}{(mean_gene_bio - baseline_gene):<15.6f}")

# =========================
# SAVE
# =========================
import pandas as pd

comparison = pd.DataFrame([{
    "unseen_gene": unseen_gene,
    "baseline_mse": baseline_mse,
    "bio_mse": mse_bio,
    "baseline_r2": baseline_r2,
    "bio_r2": r2_bio,
    "baseline_cellwise": baseline_cell,
    "bio_cellwise": mean_cell_bio,
    "baseline_genewise": baseline_gene,
    "bio_genewise": mean_gene_bio
}])

comparison.to_csv("CREB1_bio_vs_zero_comparison.csv", index=False)

print("\nSaved: CREB1_bio_vs_zero_comparison.csv")

Loaded best_geneformer_lite.pt

CREB1 ZERO-SHOT COMPARISON
Metric              Zero Vector    Bio Proxy      Delta          
----------------------------------------------------------------------
MSE                 0.092426       0.089521       0.002905       
R²                  0.738403       0.746626       0.008223       
Cell Pearson        0.860640       0.865142       0.004502       
Gene Pearson        0.000929       0.000815       -0.000114      

Saved: CREB1_bio_vs_zero_comparison.csv
